# Elasticsearch Vector Search Deep Dive — Hands-on Jupyter Notebook

> **환경:** Elastic Cloud Hosted (ECH) + GitHub Codespace  
> **ES 버전:** 9.x  
> **모델:** `.jina-embeddings-v5-text-small` (EIS), `.jina-reranker-v2-base-multilingual` (EIS)  
> **소요 시간:** Core Track ~40분 / 전체 ~120분

---

## 구성 트랙
| 트랙 | 섹션 | 소요 |
|------|------|------|
| **Core** (필수) | §1 ~ §4, §7, §9, §11 | ~40분 |
| **Advanced** | §5, §6, §8, §10, §12, §13 | ~30분 |
| **Expert** | §14, §15, §16, §17 ES\|QL | ~20분 |
| **Applied** | §20 찬봇 시나리오 | ~15분 |

---


## 0. 환경 설정 및 라이브러리 설치

In [ ]:
# 필수 라이브러리 설치
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("elasticsearch>=8.12.0", "requests", "urllib3")
print("✅ 라이브러리 설치 완료")


In [ ]:
import os, json, time, getpass
from elasticsearch import Elasticsearch, helpers
from pprint import pprint

# ─── 클러스터 연결 정보 (Codespace 환경변수 우선 사용) ───
print("🔌 Elasticsearch 클러스터 연결 정보를 입력하세요.")
es_url  = os.environ.get("ES_URL")      or input("ES URL [기본값: https://localhost:9200]: ") or "https://localhost:9200"
es_user = os.environ.get("ES_USER")     or input("Username [기본값: elastic]: ") or "elastic"
es_pass = os.environ.get("ES_PASSWORD") or getpass.getpass("Password: ")

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

es = Elasticsearch(
    es_url,
    basic_auth=(es_user, es_pass),
    verify_certs=False,
    request_timeout=60
)

info = es.info()
print(f"✅ 연결 성공: {info['name']} (ES {info['version']['number']})")


In [ ]:
# ─── Inference Endpoint 이름 상수 정의 ───
# EIS 사전 등록 모델 ID (PUT _inference 시 model_id로 사용)
EMBED_MODEL_ID   = "jina-embeddings-v5-text-small"   # EIS embedding model
RERANK_MODEL_ID  = "jina-reranker-v2-base-multilingual"  # 다국어(한국어) reranker

# Inference Endpoint 식별자 (API 호출 시 사용)
EMBED_EP   = ".jina-embeddings-v5-text-small"
RERANK_EP  = ".jina-reranker-v2-base-multilingual"

# 주 실습 인덱스
INDEX_MAIN = "vector-search-lab"

print(f"Embedding  EP : {EMBED_EP}")
print(f"Rerank     EP : {RERANK_EP}")
print(f"Main Index    : {INDEX_MAIN}")


---
## §1. Inference Endpoint 생성 (Core)

EIS(Elastic Inference Service)에서 제공되는 Jina 모델을 Inference Endpoint로 등록합니다.  
EIS 모델은 별도 API 키 없이 ECH에서 바로 사용 가능합니다.

> **사전 등록된 EIS Endpoint 확인**  
> ECH 8.15+ 환경에서는 `.jina-*` 로 시작하는 EIS endpoint가 이미 내장되어 있습니다.  
> 아래 셀에서 존재 여부를 확인하고, 없을 경우에만 생성합니다.


In [ ]:
# ─── 1-A. EIS Inference Endpoint 존재 확인 & 생성 ───
def ensure_inference_endpoint(task_type: str, endpoint_id: str, model_id: str):
    """Inference Endpoint가 없으면 생성, 있으면 스킵"""
    try:
        resp = es.inference.get(inference_id=endpoint_id)
        print(f"✅ [{endpoint_id}] 이미 존재합니다.")
        return resp
    except Exception:
        pass  # 없으면 생성

    body = {
        "service": "elastic",
        "service_settings": {"model_id": model_id}
    }
    resp = es.inference.put(task_type=task_type, inference_id=endpoint_id, body=body)
    print(f"🆕 [{endpoint_id}] 생성 완료.")
    return resp

ensure_inference_endpoint("text_embedding", EMBED_EP, EMBED_MODEL_ID)
ensure_inference_endpoint("rerank",          RERANK_EP, RERANK_MODEL_ID)


In [ ]:
# ─── 1-B. Inference Endpoint 동작 검증 ───
test_resp = es.inference.inference(
    inference_id=EMBED_EP,
    input=["Elasticsearch 벡터 검색 테스트"]
)
embeddings = test_resp.get("text_embedding", [])
if embeddings:
    dims = len(embeddings[0]["embedding"])
    print(f"✅ Embedding 생성 성공! 차원 수: {dims}d")
else:
    print("⚠️ Embedding 응답을 확인하세요:", test_resp)


In [ ]:
# ─── 1-C. 전체 Inference Endpoint 목록 확인 ───
all_eps = es.inference.get()
for ep in all_eps.get("endpoints", []):
    print(f"  [{ep.get('task_type')}] {ep.get('inference_id')}")


---
## §2. semantic_text 인덱스 생성 및 데이터 인덱싱 (Core)

`semantic_text` 타입은 자동으로 **chunking → embedding → nested 저장**을 처리합니다.  
`title` 필드는 BM25 lexical 검색용으로 함께 유지합니다.


In [ ]:
# ─── 2-A. 인덱스 생성 (이미 있으면 삭제 후 재생성) ───
def recreate_index(index_name: str, mappings: dict, settings: dict = None):
    """인덱스를 클린하게 재생성합니다."""
    if es.indices.exists(index=index_name):
        es.indices.delete(index=index_name)
        print(f"🗑️  [{index_name}] 삭제 완료")
    body = {"mappings": mappings}
    if settings:
        body["settings"] = settings
    es.indices.create(index=index_name, body=body)
    print(f"✅ [{index_name}] 생성 완료")

recreate_index(INDEX_MAIN, {
    "properties": {
        "title":    {"type": "text",  "analyzer": "standard"},
        "content":  {"type": "semantic_text", "inference_id": EMBED_EP},
        "category": {"type": "keyword"}
    }
})


In [ ]:
# ─── 2-B. 한국어 샘플 데이터 인덱싱 ───
DOCS = [
    {"_id": "1", "title": "HNSW 알고리즘의 원리",
     "content": "HNSW(Hierarchical Navigable Small World)는 벡터 검색에서 가장 널리 사용되는 ANN 알고리즘입니다. 계층적 그래프 구조를 통해 O(log N) 시간 복잡도로 근사 최근접 이웃을 찾습니다. 상위 레이어에서 장거리 점프로 탐색을 시작하고, 하위 레이어로 내려가며 정밀한 검색을 수행합니다.",
     "category": "algorithm"},
    {"_id": "2", "title": "벡터 양자화와 BBQ",
     "content": "BBQ(Better Binary Quantization)는 float32 벡터를 1-bit로 압축하면서도 높은 recall을 유지하는 기술입니다. centroid 기준 정규화, corrective factor 저장, asymmetric quantization을 통해 naive binary 대비 극적인 품질 향상을 달성합니다. ES 9.1 이상에서 384차원 이상 벡터에 기본 적용됩니다.",
     "category": "optimization"},
    {"_id": "3", "title": "Hybrid Search 구현 가이드",
     "content": "Hybrid Search는 BM25 lexical 검색과 벡터 semantic 검색을 결합하여 단일 방식의 한계를 극복합니다. RRF는 순위 기반으로 간편하게 결합하고, Linear retriever는 정규화된 점수의 가중합으로 더 정밀한 제어가 가능합니다. Reranker를 추가하면 cross-encoder의 양방향 attention으로 최종 정밀도를 극대화할 수 있습니다.",
     "category": "search"},
    {"_id": "4", "title": "RAG 파이프라인 설계 원칙",
     "content": "RAG(Retrieval-Augmented Generation)에서 검색 품질은 LLM 출력 품질을 직접 결정합니다. 잘못된 chunk가 context로 전달되면 hallucination이 발생합니다. min_score cutoff로 저품질 결과를 필터하고, context 길이를 8K 토큰 이내로 유지하면 대부분의 RAG 실패 모드를 예방할 수 있습니다.",
     "category": "rag"},
    {"_id": "5", "title": "임베딩 모델 선택 가이드",
     "content": "한국어 환경에서는 다국어를 지원하는 임베딩 모델이 필수입니다. ELSER는 영어 전용이므로 적합하지 않습니다. jina-embeddings-v5-text-small은 119개 언어를 지원하며 MMTEB 벤치마크에서 동급 모델 중 최고 성능을 보입니다. 32K 토큰 컨텍스트 윈도우와 Matryoshka 차원 축소를 지원합니다.",
     "category": "model"},
    {"_id": "6", "title": "Chunking 전략과 검색 품질",
     "content": "Chunking은 RAG 파이프라인에서 검색 품질의 50%를 결정합니다. Recursive splitting(512 토큰, 10-20% overlap)이 현재 가장 안정적인 기본 전략입니다. Semantic chunking은 recall을 높일 수 있지만 비용이 높고 fragment가 작아지는 문제가 있습니다. Late chunking은 긴 문서에서 대명사 참조를 유지하는 데 효과적입니다.",
     "category": "chunking"},
    {"_id": "7", "title": "Elasticsearch의 벡터 검색 아키텍처",
     "content": "Elasticsearch는 Lucene의 HNSW 구현을 기반으로 벡터 검색을 제공합니다. 각 Lucene segment에 독립적인 HNSW 그래프가 생성되며, segment merge 시 그래프가 재구성됩니다. ACORN-1 알고리즘은 filtered kNN 검색에서 neighbor의 neighbor까지 탐색을 확장하여 최대 5배 속도 향상을 제공합니다.",
     "category": "architecture"},
    {"_id": "8", "title": "API 응답 시간 최적화 방법론",
     "content": "엔드포인트 레이턴시를 줄이기 위해서는 여러 계층의 최적화가 필요합니다. 네트워크 레벨에서는 CDN과 엣지 캐싱을 활용하고, 애플리케이션 레벨에서는 connection pooling과 비동기 처리를 적용합니다. 데이터베이스 레벨에서는 인덱싱 전략과 쿼리 최적화가 핵심입니다. 벡터 검색의 경우 BBQ 양자화와 num_candidates 조정이 가장 효과적인 최적화 방법입니다.",
     "category": "performance"},
]

actions = [{"_index": INDEX_MAIN, "_id": doc["_id"],
            "_source": {k: v for k, v in doc.items() if k != "_id"}}
           for doc in DOCS]

success, errors = helpers.bulk(es, actions)
print(f"✅ {success}건 인덱싱 완료. 오류: {errors}")
print("⏳ EIS 임베딩 생성 대기 중 (약 15~30초)...")
time.sleep(20)


In [ ]:
# ─── 2-C. 인덱싱 상태 확인 ───
count = es.count(index=INDEX_MAIN)
print(f"총 문서 수: {count['count']}건")

sample = es.search(index=INDEX_MAIN, size=2, source=["title", "category"])
for hit in sample["hits"]["hits"]:
    print(f"  [{hit['_id']}] {hit['_source']['title']} ({hit['_source']['category']})")


---
## §3. semantic_text 내부 구조 확인 (Core)

`semantic_text`는 내부적으로 **자동 chunking → embedding → nested 저장**을 수행합니다.  
이 섹션에서 그 구조를 직접 확인합니다.


In [ ]:
# ─── 3-A. 자동 생성된 매핑 확인 ───
mapping = es.indices.get_mapping(index=INDEX_MAIN)
content_mapping = mapping[INDEX_MAIN]["mappings"]["properties"]["content"]
print("content 필드 매핑:")
print(json.dumps(content_mapping, indent=2, ensure_ascii=False))


In [ ]:
# ─── 3-B. 저장된 chunk 구조 직접 확인 ───
# ES 9.x: _source에서 dot-notation 필터링 대신 전체 source를 가져온 뒤 Python에서 접근
resp = es.search(
    index=INDEX_MAIN,
    size=1,
    query={"term": {"_id": "1"}},
    source=True  # 전체 _source 반환
)
hit = resp["hits"]["hits"][0]["_source"]
print(f"제목: {hit['title']}")

# content 필드 안의 inference 구조 접근
content_field = hit.get("content", {})
# semantic_text는 {"text": "원문", "inference": {"chunks": [...]}} 구조로 저장
if isinstance(content_field, dict):
    chunks = content_field.get("inference", {}).get("chunks", [])
else:
    # 일부 버전: content가 문자열로 저장되고 inference는 별도 메타필드
    chunks = []
    print("  ℹ️  이 클러스터에서는 inference 데이터가 _source에 포함되지 않습니다.")
    print("     (semantic_text의 embedding은 내부 저장소에서만 검색에 사용됩니다)")

print(f"Chunk 수: {len(chunks)}")
for i, chunk in enumerate(chunks):
    txt = chunk.get("text", "")[:60]
    emb = chunk.get("embeddings", [])
    emb_len = len(emb) if isinstance(emb, list) else "(벡터는 _source 미포함)"
    print(f"  Chunk {i+1}: '{txt}...' | embedding 차원: {emb_len}")

if not chunks:
    print("\n💡 참고: ES 9.x에서 semantic_text의 내부 chunk/embedding은")
    print("   _source에 노출되지 않고 Lucene 내부에 저장됩니다.")
    print("   실제 매칭된 chunk 텍스트는 아래 3-C inner_hits로 확인합니다.")


In [ ]:
# ─── 3-C. inner_hits로 매칭된 chunk 확인 ───
# ES 9.x: semantic query의 inner_hits는 query 안이 아닌 별도 방식으로 접근
# semantic_text는 내부적으로 nested 구조 → nested query로 inner_hits 확인

# 방법 1: semantic query로 검색 후 해당 문서의 chunk 텍스트 직접 조회
resp = es.search(
    index=INDEX_MAIN,
    size=3,
    source=["title", "content"],
    query={
        "semantic": {
            "field": "content",
            "query": "벡터 압축 기술"
        }
    }
)
print("=== [Query DSL] Semantic 검색 결과 + chunk 접근 ===")
for hit in resp["hits"]["hits"]:
    title = hit["_source"]["title"]
    print(f"\n📄 [{hit['_score']:.4f}] {title}")
    # content 필드에서 원문 텍스트 추출
    content_val = hit["_source"].get("content", "")
    if isinstance(content_val, dict):
        # semantic_text가 dict로 저장된 경우 (원문 + inference)
        text = content_val.get("text", "")
        chunks = content_val.get("inference", {}).get("chunks", [])
        if chunks:
            for j, ch in enumerate(chunks[:2]):
                print(f"  ↳ chunk {j+1}: {ch.get('text','')[:80]}...")
        elif text:
            print(f"  ↳ 원문: {text[:100]}...")
    elif isinstance(content_val, str):
        print(f"  ↳ 원문: {content_val[:100]}...")

print()
print("💡 ES 9.x에서 semantic query의 inner_hits는 query DSL 내부가 아닌")
print("   knn retriever 또는 _search API의 top-level inner_hits로 지원됩니다.")

# 방법 2: knn retriever로 inner_hits 접근 (semantic_text 내부 nested chunk 확인)
print("\n=== [Query DSL] kNN retriever로 매칭 chunk 확인 ===")
resp2 = es.search(
    index=INDEX_MAIN,
    size=3,
    source=["title"],
    knn={
        "field": "content.inference.chunks.embeddings",
        "query_vector_builder": {
            "text_embedding": {
                "model_id": EMBED_EP,
                "model_text": "벡터 압축 기술"
            }
        },
        "k": 3,
        "num_candidates": 20,
        "inner_hits": {
            "size": 2,
            "_source": ["content.inference.chunks.text"],
            "name": "matched_chunks"
        }
    }
)
for hit in resp2["hits"]["hits"]:
    print(f"\n📄 [{hit['_score']:.4f}] {hit['_source']['title']}")
    inner = (hit.get("inner_hits", {})
                .get("matched_chunks", {})
                .get("hits", {})
                .get("hits", []))
    for ih in inner:
        chunk_src = ih.get("_source", {})
        # chunk 텍스트 경로
        chunk_text = ""
        content_inf = chunk_src.get("content", {})
        if isinstance(content_inf, dict):
            chunks_list = content_inf.get("inference", {}).get("chunks", [])
            if chunks_list:
                chunk_text = chunks_list[0].get("text", "")
        print(f"  ↳ matched chunk: {chunk_text[:80] if chunk_text else '(텍스트 없음)'}...")
    if not inner:
        print("  ↳ (inner_hits 없음 — 필드 경로 확인 필요)")


In [ ]:
# ─── 3-D. ES|QL Semantic Search (기본) ───
# ES|QL WHERE 절에 semantic_text 필드 사용 → 자동으로 semantic query 실행
esql_query = """
FROM vector-search-lab METADATA _score
| WHERE content:"벡터 압축 기술"
| KEEP title, category, _score
| SORT _score DESC
| LIMIT 5
"""
esql_resp = es.esql.query(query=esql_query)
print("=== [ES|QL] Semantic 검색 결과 ===")
cols = [c["name"] for c in esql_resp["columns"]]
print(f"  {'제목':<30} {'카테고리':<15} {'score'}")
print("  " + "-"*60)
for row in esql_resp["values"]:
    row_dict = dict(zip(cols, row))
    print(f"  {str(row_dict.get('title','')):<30} {str(row_dict.get('category','')):<15} {row_dict.get('_score', 0):.4f}")


In [ ]:
# ─── 3-E. Chunking 설정 변경 테스트 인덱스 (생성 → 확인 → 삭제) ───
CHUNK_TEST_IDX = "vector-search-lab-chunk-test"
recreate_index(CHUNK_TEST_IDX, {
    "properties": {
        "content": {
            "type": "semantic_text",
            "inference_id": EMBED_EP,
            "chunking_settings": {
                "strategy": "sentence",
                "max_chunk_size": 150,
                "sentence_overlap": 1
            }
        }
    }
})
mapping_ct = es.indices.get_mapping(index=CHUNK_TEST_IDX)
chunk_settings = (mapping_ct[CHUNK_TEST_IDX]["mappings"]
                  .get("properties", {})
                  .get("content", {})
                  .get("chunking_settings", {}))
print("chunking_settings:", json.dumps(chunk_settings, indent=2))
es.indices.delete(index=CHUNK_TEST_IDX)
print(f"🗑️  [{CHUNK_TEST_IDX}] 정리 완료")


---
## §4. Semantic Query 실행 (Core)

Query DSL과 ES|QL 두 방식으로 Semantic Query를 실행하고 결과를 비교합니다.


In [ ]:
# ─── 4-A. 기본 semantic query ─── Query DSL
resp = es.search(
    index=INDEX_MAIN,
    size=5,
    source=["title", "category"],
    query={"semantic": {"field": "content", "query": "벡터 검색 속도를 높이려면 어떻게 해야 하나요?"}}
)
print("=== [Query DSL] Semantic Query ===")
for h in resp["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']} ({h['_source']['category']})")


In [ ]:
# ─── 4-A. 기본 semantic query ─── ES|QL
esql_resp = es.esql.query(query="""
FROM vector-search-lab METADATA _score
| WHERE content:"벡터 검색 속도를 높이려면 어떻게 해야 하나요?"
| KEEP title, category, _score
| SORT _score DESC
| LIMIT 5
""")
print("=== [ES|QL] Semantic Query ===")
cols = [c["name"] for c in esql_resp["columns"]]
for row in esql_resp["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


In [ ]:
# ─── 4-B. 다국어 semantic query (영어 쿼리 → 한국어 문서 매칭) ───
# Query DSL
resp_en = es.search(
    index=INDEX_MAIN, size=3, source=["title"],
    query={"semantic": {"field": "content", "query": "How to optimize vector search performance?"}}
)
print("=== [Query DSL] 영어 쿼리 → 한국어 문서 매칭 ===")
for h in resp_en["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']}")

# ES|QL
esql_en = es.esql.query(query="""
FROM vector-search-lab METADATA _score
| WHERE content:"How to optimize vector search performance?"
| KEEP title, _score
| SORT _score DESC
| LIMIT 3
""")
print("\n=== [ES|QL] 영어 쿼리 → 한국어 문서 매칭 ===")
cols = [c["name"] for c in esql_en["columns"]]
for row in esql_en["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')}")


In [ ]:
# ─── 4-C. BM25 vs Semantic 비교 ─── 같은 의도, 다른 단어
# ⚠️ 설계 포인트:
# - content 필드는 semantic_text 타입 → match 쿼리도 내부적으로 semantic으로 처리됨
# - BM25 비교는 반드시 text 타입 필드(title)로 해야 함
# - 이것이 "title=BM25용, content=Semantic용"으로 필드를 분리하는 이유

COMPARE_QUERY = "엔드포인트 레이턴시 개선"

print(f"=== BM25: match on title (text type) — '{COMPARE_QUERY}' ===")
bm25 = es.search(
    index=INDEX_MAIN, size=5, source=["title"],
    query={"match": {"title": COMPARE_QUERY}}
)
hits_bm25 = bm25["hits"]["hits"]
if hits_bm25:
    for h in hits_bm25:
        print(f"  [{h['_score']:.4f}] {h['_source']['title']}")
else:
    print("  (매칭 없음 — title에 해당 단어 없음)")

print(f"\n=== Semantic: match on content (semantic_text type) — '{COMPARE_QUERY}' ===")
sem = es.search(
    index=INDEX_MAIN, size=5, source=["title"],
    query={"semantic": {"field": "content", "query": COMPARE_QUERY}}
)
for h in sem["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']}")

print()
print("💡 핵심 포인트:")
print("   • BM25(title match): '엔드포인트', '레이턴시' 단어가 title에 없으면 0건")
print("   • Semantic(content): 의미적으로 관련 있는 문서(API 최적화, 벡터 최적화 등) 반환")
print("   • semantic_text 필드에 match 쿼리를 보내면 자동으로 semantic 검색으로 처리됨")
print("   → 이것이 Hybrid Search에서 BM25는 title, Semantic은 content를 쓰는 이유")


---
## §5. dense_vector 인덱스 — 세밀 제어 (Advanced)

`semantic_text`와 달리 `dense_vector`는 HNSW 파라미터(`m`, `ef_construction`),  
양자화 방식(`bbq_hnsw`, `hnsw`)을 직접 제어할 수 있습니다.


In [ ]:
# ─── 5-A. dense_vector 인덱스 생성 (BBQ + HNSW 명시) ───
CUSTOM_IDX = "vector-search-lab-custom"
recreate_index(CUSTOM_IDX, {
    "properties": {
        "title":         {"type": "text"},
        "content_text":  {"type": "text"},
        "content_vector": {
            "type": "dense_vector",
            "dims": 1024,
            "similarity": "cosine",
            "index_options": {
                "type": "bbq_hnsw",   # BBQ 양자화 + HNSW (ES 9.1+ 384d 이상 기본값)
                "m": 16,              # 노드당 연결 수 (기본값)
                "ef_construction": 100  # 그래프 구축 탐색 범위 (기본값)
            }
        }
    }
})


In [ ]:
# ─── 5-B. Inference API로 임베딩 생성 후 인덱싱 ───
sample_texts = [
    "HNSW 알고리즘은 벡터 검색의 핵심입니다.",
    "BBQ 양자화로 벡터를 32배 압축할 수 있습니다.",
    "Hybrid Search는 BM25와 벡터 검색을 결합합니다."
]
embed_resp = es.inference.inference(
    inference_id=EMBED_EP,
    input=sample_texts
)
embeddings_list = [e["embedding"] for e in embed_resp.get("text_embedding", [])]
print(f"✅ {len(embeddings_list)}개 임베딩 생성, 차원: {len(embeddings_list[0])}d")

# 인덱싱
actions = []
for i, (txt, vec) in enumerate(zip(sample_texts, embeddings_list), start=1):
    actions.append({
        "_index": CUSTOM_IDX, "_id": str(i),
        "_source": {"title": f"샘플-{i}", "content_text": txt, "content_vector": vec}
    })
helpers.bulk(es, actions)
es.indices.refresh(index=CUSTOM_IDX)
print(f"✅ {len(actions)}건 인덱싱 완료")


In [ ]:
# ─── 5-C. 인덱스 정리 ───
es.indices.delete(index=CUSTOM_IDX)
print(f"🗑️  [{CUSTOM_IDX}] 정리 완료")


---
## §6. kNN Query + _explain (Advanced)

`knn` API로 벡터 검색을 실행하고, `explain: true`로 score 계산 과정을 분석합니다.


In [ ]:
# ─── 6-A. kNN Query — query_vector_builder 방식 ───
# Query DSL
resp_knn = es.search(
    index=INDEX_MAIN,
    source=["title", "category"],
    knn={
        "field": "content.inference.chunks.embeddings",
        "query_vector_builder": {
            "text_embedding": {
                "model_id": EMBED_EP,
                "model_text": "벡터 압축 기술"
            }
        },
        "k": 5,
        "num_candidates": 50
    }
)
print("=== [Query DSL] kNN 검색 — num_candidates=50 ===")
for h in resp_knn["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']}")


In [ ]:
# ─── 6-A. kNN Query ─── ES|QL KNN 함수
esql_knn = es.esql.query(query="""
FROM vector-search-lab METADATA _score
| WHERE KNN(content, "벡터 압축 기술", 5)
| KEEP title, category, _score
| SORT _score DESC
| LIMIT 5
""")
print("=== [ES|QL] KNN 함수 ===")
cols = [c["name"] for c in esql_knn["columns"]]
for row in esql_knn["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


In [ ]:
# ─── 6-B. _explain으로 score 분해 ───
resp_explain = es.search(
    index=INDEX_MAIN,
    explain=True,
    size=2,
    source=["title"],
    query={"semantic": {"field": "content", "query": "벡터 압축 기술"}}
)
print("=== _explain score 분해 (첫 번째 결과) ===")
hit = resp_explain["hits"]["hits"][0]
print(f"문서: {hit['_source']['title']} | score: {hit['_score']:.4f}")
print("Explanation (요약):")
exp = hit.get("_explanation", {})
print(f"  description: {exp.get('description', '')[:100]}")
for detail in exp.get("details", [])[:2]:
    print(f"  └─ {detail.get('description','')[:80]} = {detail.get('value',0):.4f}")


---
## §7. Hybrid Search — RRF Retriever (Core)

**BM25 + Semantic**을 **RRF(Reciprocal Rank Fusion)**로 결합합니다.  
RRF는 각 retriever의 순위만 고려하여 score를 결합합니다 (`1/(rank + k)`).


In [ ]:
QUERY_TEXT = "벡터 검색 성능을 개선하는 방법"

# ─── 7-A. RRF Retriever ─── Query DSL
resp_rrf = es.search(
    index=INDEX_MAIN, size=5, source=["title", "category"],
    retriever={
        "rrf": {
            "retrievers": [
                {"standard": {"query": {"match": {"title": "벡터 검색 최적화"}}}},
                {"standard": {"query": {"semantic": {"field": "content", "query": QUERY_TEXT}}}}
            ],
            "rank_constant": 60,
            "rank_window_size": 50
        }
    }
)
print("=== [Query DSL] RRF Hybrid Search ===")
for h in resp_rrf["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']} ({h['_source']['category']})")


In [ ]:
# ─── 7-A. RRF ─── ES|QL FORK + FUSE
esql_fork_fuse = es.esql.query(query=f"""
FROM vector-search-lab METADATA _score
| FORK
    (WHERE title:"벡터 검색 최적화" | SORT _score DESC | LIMIT 10),
    (WHERE KNN(content, "{QUERY_TEXT}", 10))
| FUSE
| KEEP title, category, _score
| SORT _score DESC
| LIMIT 5
""")
print("=== [ES|QL] FORK + FUSE (RRF) ===")
cols = [c["name"] for c in esql_fork_fuse["columns"]]
for row in esql_fork_fuse["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


In [ ]:
# ─── 7-B. Weighted RRF ─── Query DSL
resp_wrrf = es.search(
    index=INDEX_MAIN, size=5, source=["title"],
    retriever={
        "rrf": {
            "retrievers": [
                {
                    "retriever": {"standard": {"query": {"match": {"title": "벡터 검색 최적화"}}}},
                    "weight": 0.3
                },
                {
                    "retriever": {"standard": {"query": {"semantic": {"field": "content", "query": QUERY_TEXT}}}},
                    "weight": 0.7
                }
            ],
            "rank_constant": 60,
            "rank_window_size": 50
        }
    }
)
print("=== [Query DSL] Weighted RRF (BM25:0.3, Semantic:0.7) ===")
for h in resp_wrrf["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']}")


In [ ]:
# ─── 7-B. Weighted RRF ─── ES|QL multi-field query
esql_mf = es.esql.query(query="""
FROM vector-search-lab METADATA _score
| FORK
    (WHERE title:"벡터 검색 최적화" | SORT _score DESC | LIMIT 10),
    (WHERE KNN(content, "벡터 검색 성능을 개선하는 방법", 10))
| FUSE normalizer=minmax
| KEEP title, category, _score
| SORT _score DESC
| LIMIT 5
""")
print("=== [ES|QL] FUSE normalizer=minmax (Linear 방식) ===")
cols = [c["name"] for c in esql_mf["columns"]]
for row in esql_mf["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


---
## §8. Hybrid Search — Linear Retriever (Advanced)

**Linear retriever**는 MinMax 정규화 후 가중합으로 score를 결합합니다.  
RRF가 순위만 고려하는 반면, Linear는 score의 실제 크기 차이를 반영합니다.


In [ ]:
# ─── 8. Linear Retriever ─── Query DSL
resp_linear = es.search(
    index=INDEX_MAIN, size=5, source=["title"],
    retriever={
        "linear": {
            "retrievers": [
                {
                    "retriever": {"standard": {"query": {"match": {"title": "벡터 검색 최적화"}}}},
                    "weight": 0.3
                },
                {
                    "retriever": {"standard": {"query": {"semantic": {"field": "content", "query": QUERY_TEXT}}}},
                    "weight": 0.7
                }
            ],
            "normalizer": "minmax"
        }
    }
)
print("=== [Query DSL] Linear Retriever (MinMax 정규화, BM25:0.3, Semantic:0.7) ===")
for h in resp_linear["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']}")
print("\n💡 RRF 결과와 순위 차이를 비교해보세요 (score 크기 차이가 반영됨)")


---
## §9. Semantic Reranking 파이프라인 (Core)

**RRF → Reranker** 2단계 파이프라인입니다.  
Reranker는 cross-encoder로 query+document를 동시에 읽어 정밀도를 극대화합니다.


In [ ]:
# ─── 9-A. RRF + Reranker ─── Query DSL
resp_rerank = es.search(
    index=INDEX_MAIN, size=5, source=["title", "category"],
    retriever={
        "text_similarity_reranker": {
            "retriever": {
                "rrf": {
                    "retrievers": [
                        {"standard": {"query": {"match": {"title": "검색 성능 최적화"}}}},
                        {"standard": {"query": {"semantic": {"field": "content", "query": "검색 성능을 향상시키는 실질적인 방법"}}}}
                    ],
                    "rank_window_size": 50
                }
            },
            "field": "content",
            "inference_id": RERANK_EP,
            "inference_text": "검색 성능을 향상시키는 실질적인 방법",
            "rank_window_size": 10
        }
    }
)
print("=== [Query DSL] RRF + Reranker ===")
for h in resp_rerank["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']} ({h['_source']['category']})")


In [ ]:
# ─── 9-A. RRF + Reranker ─── ES|QL FORK + FUSE + RERANK
q9a = (
    "FROM vector-search-lab METADATA _score\n"
    "| FORK\n"
    '    (WHERE title:\"검색 성능 최적화\" | SORT _score DESC | LIMIT 10),\n'
    '    (WHERE KNN(content, \"검색 성능을 향상시키는 실질적인 방법\", 10))\n'
    "| FUSE\n"
    f'| RERANK \"검색 성능을 향상시키는 실질적인 방법\" ON content WITH {{"inference_id": "{RERANK_EP}"}}\n'
    "| KEEP title, category, _score\n"
    "| LIMIT 5"
)
esql_rerank = es.esql.query(query=q9a)
print("=== [ES|QL] FORK + FUSE + RERANK ===")
cols = [c["name"] for c in esql_rerank["columns"]]
for row in esql_rerank["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")
print("\n💡 Query DSL과 ES|QL 결과의 순위를 비교해보세요")


---
## §10. min_score Cutoff 적용 (Advanced)

`min_score`는 reranker 이후 최종 score에 적용됩니다.  
RAG에서 낮은 관련도 문서를 LLM context에서 차단하는 핵심 설정입니다.


In [ ]:
# ─── 10. min_score cutoff ─── Query DSL
for threshold in [0.1, 0.3, 0.5, 0.7]:
    resp_ms = es.search(
        index=INDEX_MAIN, size=5, source=["title"],
        min_score=threshold,
        retriever={
            "text_similarity_reranker": {
                "retriever": {
                    "rrf": {
                        "retrievers": [
                            {"standard": {"query": {"match": {"title": "검색 최적화"}}}},
                            {"standard": {"query": {"semantic": {"field": "content", "query": "벡터 검색 성능 개선"}}}}
                        ],
                        "rank_window_size": 50
                    }
                },
                "field": "content",
                "inference_id": RERANK_EP,
                "inference_text": "벡터 검색 성능 개선",
                "rank_window_size": 10
            }
        }
    )
    total = resp_ms["hits"]["total"]["value"]
    print(f"min_score={threshold}: {total}건 반환")

print("\n💡 min_score가 높을수록 반환 문서 수 감소 — RAG에서 hallucination 차단")


---
## §11. 전체 3단계 파이프라인 (Core)

**BM25 + Semantic → RRF → Reranker → min_score**  
단일 `_search` API 호출로 완성되는 프로덕션 레벨 파이프라인입니다.


In [ ]:
PIPELINE_QUERY = "벡터 검색 성능을 개선하고 비용을 줄이는 방법"

# ─── 11. 3단계 파이프라인 ─── Query DSL
resp_full = es.search(
    index=INDEX_MAIN,
    size=5,
    source=["title", "category"],
    min_score=0.3,
    retriever={
        "text_similarity_reranker": {
            "retriever": {
                "rrf": {
                    "retrievers": [
                        {"standard": {"query": {"multi_match": {
                            "query": "벡터 검색 최적화 방법",
                            "fields": ["title^2", "content"]
                        }}}},
                        {"standard": {"query": {"semantic": {
                            "field": "content",
                            "query": PIPELINE_QUERY
                        }}}}
                    ],
                    "rank_constant": 60,
                    "rank_window_size": 50
                }
            },
            "field": "content",
            "inference_id": RERANK_EP,
            "inference_text": PIPELINE_QUERY,
            "rank_window_size": 20
        }
    }
)
print("=== [Query DSL] 전체 파이프라인: multi_match + Semantic → RRF → Reranker ===")
print(f"파이프라인: BM25(title×2+content) + Semantic → RRF(top-50) → Reranker(top-20) → min_score≥0.3")
print()
for h in resp_full["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']} ({h['_source']['category']})")


In [ ]:
# ─── 11. 3단계 파이프라인 ─── ES|QL FORK + FUSE + RERANK
q11 = (
    "FROM vector-search-lab METADATA _score\n"
    "| FORK\n"
    '    (WHERE title:\"벡터 검색 최적화\" | SORT _score DESC | LIMIT 20),\n'
    f'    (WHERE KNN(content, \"{PIPELINE_QUERY}\", 20))\n'
    "| FUSE\n"
    f'| RERANK \"{PIPELINE_QUERY}\" ON CONCAT(title, \" - \", content) WITH {{"inference_id": "{RERANK_EP}"}}\n'
    "| KEEP title, category, _score\n"
    "| LIMIT 5"
)
esql_full = es.esql.query(query=q11)
print("=== [ES|QL] 전체 파이프라인: FORK + FUSE + RERANK (title+content 조합) ===")
cols = [c["name"] for c in esql_full["columns"]]
for row in esql_full["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


---
## §12. Filter + Semantic 조합 (Advanced)

실무에서 가장 많이 사용하는 패턴: 카테고리/권한/날짜 filter + semantic 검색


In [ ]:
# ─── 12-A. bool.filter + semantic ─── Query DSL
resp_filter_sem = es.search(
    index=INDEX_MAIN, size=3, source=["title", "category"],
    query={
        "bool": {
            "must": {"semantic": {"field": "content", "query": "검색 성능을 개선하는 방법"}},
            "filter": {"term": {"category": "optimization"}}
        }
    }
)
print("=== [Query DSL] filter(category=optimization) + Semantic ===")
for h in resp_filter_sem["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']} ({h['_source']['category']})")


In [ ]:
# ─── 12-A. filter + Semantic ─── ES|QL
esql_filter = es.esql.query(query="""
FROM vector-search-lab METADATA _score
| WHERE category == "optimization"
| FORK
    (WHERE title:"검색 최적화" | SORT _score DESC | LIMIT 10),
    (WHERE KNN(content, "검색 성능을 개선하는 방법", 10))
| FUSE
| KEEP title, category, _score
| SORT _score DESC
| LIMIT 5
""")
print("=== [ES|QL] pre-filter(category=optimization) + FORK + FUSE ===")
cols = [c["name"] for c in esql_filter["columns"]]
for row in esql_filter["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


In [ ]:
# ─── 12-B. RRF retriever 레벨 filter ─── Query DSL
resp_rrf_filter = es.search(
    index=INDEX_MAIN, size=5, source=["title", "category"],
    retriever={
        "rrf": {
            "retrievers": [
                {"standard": {"query": {"match": {"title": "검색 최적화"}}}},
                {"standard": {"query": {"semantic": {"field": "content", "query": "검색 성능 개선"}}}}
            ],
            "filter": {"term": {"category": "search"}},
            "rank_window_size": 50
        }
    }
)
print("=== [Query DSL] RRF retriever filter(category=search) ===")
for h in resp_rrf_filter["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']} ({h['_source']['category']})")


---
## §13. Filter + kNN — ACORN-1 알고리즘 체험 (Advanced)

ES 9.x의 **ACORN-1 알고리즘**은 filtered kNN에서 neighbor의 neighbor까지 탐색을 확장하여  
filter로 인한 recall 손실을 방지합니다.


In [ ]:
# ─── 13. kNN + filter (ACORN-1 자동 적용) ─── Query DSL
resp_acorn = es.search(
    index=INDEX_MAIN,
    source=["title", "category"],
    knn={
        "field": "content.inference.chunks.embeddings",
        "query_vector_builder": {
            "text_embedding": {
                "model_id": EMBED_EP,
                "model_text": "벡터 압축과 저장 최적화"
            }
        },
        "k": 5,
        "num_candidates": 50,
        "filter": {"term": {"category": "optimization"}}
    }
)
print("=== [Query DSL] kNN + filter=optimization (ACORN-1 자동 적용) ===")
for h in resp_acorn["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']} ({h['_source']['category']})")
print("\n💡 filter 비율이 높을수록 ACORN-1의 효과가 큽니다.")


---
## §14. Multi-field Query — 간편 Hybrid (Expert)

ES 9.x 신기능: `retriever`에 `query`와 `fields`만 지정하면  
lexical 필드 → BM25, semantic_text 필드 → semantic query가 **자동 분류**됩니다.


In [ ]:
# ─── 14-A. RRF multi-field query ─── Query DSL
resp_mf_rrf = es.search(
    index=INDEX_MAIN, size=5, source=["title", "category"],
    retriever={
        "rrf": {
            "query": "벡터 검색 최적화",
            "fields": ["title", "content"]
        }
    }
)
print("=== [Query DSL] RRF multi-field (title→BM25, content→Semantic 자동) ===")
for h in resp_mf_rrf["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']} ({h['_source']['category']})")


In [ ]:
# ─── 14-B. Linear multi-field query ─── Query DSL
resp_mf_linear = es.search(
    index=INDEX_MAIN, size=5, source=["title", "category"],
    retriever={
        "linear": {
            "query": "벡터 검색 최적화",
            "fields": ["title^2", "content"],
            "normalizer": "minmax"
        }
    }
)
print("=== [Query DSL] Linear multi-field (title^2 가중치, MinMax 정규화) ===")
for h in resp_mf_linear["hits"]["hits"]:
    print(f"  [{h['_score']:.4f}] {h['_source']['title']} ({h['_source']['category']})")
print("\n💡 가장 간단한 프로덕션 hybrid search 코드입니다.")


---
## §15. BBQ 저장 크기 비교 (Expert)

**float32 HNSW** vs **BBQ HNSW** 인덱스의 실제 저장 크기를 비교합니다.  
BBQ는 float32 대비 최대 32배 압축 효과를 제공합니다.


In [ ]:
# ─── 15. BBQ 저장 크기 비교 ───
FLOAT32_IDX = "vector-search-lab-float32"
BBQ_IDX     = "vector-search-lab-bbq"

# float32 인덱스
recreate_index(FLOAT32_IDX, {
    "properties": {
        "title": {"type": "text"},
        "vec":   {"type": "dense_vector", "dims": 1024, "similarity": "cosine",
                  "index_options": {"type": "hnsw"}}
    }
})
# BBQ 인덱스
recreate_index(BBQ_IDX, {
    "properties": {
        "title": {"type": "text"},
        "vec":   {"type": "dense_vector", "dims": 1024, "similarity": "cosine",
                  "index_options": {"type": "bbq_hnsw"}}
    }
})

# 공통 데이터: Inference API로 임베딩 생성
texts = [f"Elasticsearch 벡터 검색 테스트 문서 {i}번입니다." for i in range(1, 51)]
embed_resp = es.inference.inference(inference_id=EMBED_EP, input=texts)
vecs = [e["embedding"] for e in embed_resp.get("text_embedding", [])]

for idx_name in [FLOAT32_IDX, BBQ_IDX]:
    actions = [{"_index": idx_name, "_id": str(i+1),
                "_source": {"title": texts[i], "vec": vecs[i]}}
               for i in range(len(vecs))]
    helpers.bulk(es, actions)
    es.indices.forcemerge(index=idx_name, max_num_segments=1)  # 정확한 크기 비교
    time.sleep(3)

# 크기 비교
for idx_name in [FLOAT32_IDX, BBQ_IDX]:
    stats = es.indices.stats(index=idx_name, metric="store")
    size_bytes = stats["indices"][idx_name]["total"]["store"]["size_in_bytes"]
    label = "float32 HNSW" if idx_name == FLOAT32_IDX else "BBQ   HNSW"
    print(f"  [{label}] {size_bytes:,} bytes ({size_bytes/1024:.1f} KB)")

# 정리
for idx_name in [FLOAT32_IDX, BBQ_IDX]:
    es.indices.delete(index=idx_name)
    print(f"🗑️  [{idx_name}] 정리 완료")


---
## §16. 인덱스 Alias + 무중단 모델 교체 (Expert)

Alias를 사용하면 재인덱싱 없이 atomic하게 모델을 교체할 수 있습니다.  
애플리케이션은 alias만 참조하므로 다운타임이 없습니다.


In [ ]:
# ─── 16-A. 현재 인덱스에 alias 추가 ───
ALIAS_NAME = "vector-search-active"
es.indices.put_alias(index=INDEX_MAIN, name=ALIAS_NAME)
print(f"✅ alias '{ALIAS_NAME}' → '{INDEX_MAIN}' 설정")

# alias로 검색 (동일하게 동작)
resp_alias = es.search(
    index=ALIAS_NAME, size=2, source=["title"],
    query={"match_all": {}}
)
print(f"alias 검색 결과: {resp_alias['hits']['total']['value']}건")


In [ ]:
# ─── 16-B. 새 인덱스 생성 (v2 — 동일 모델, 실전에서는 신규 모델로 교체) ───
V2_IDX = "vector-search-lab-v2"
recreate_index(V2_IDX, {
    "properties": {
        "title":    {"type": "text", "analyzer": "standard"},
        "content":  {"type": "semantic_text", "inference_id": EMBED_EP},
        "category": {"type": "keyword"}
    }
})
# reindex
es.reindex(
    body={"source": {"index": INDEX_MAIN}, "dest": {"index": V2_IDX}},
    wait_for_completion=True
)
es.indices.refresh(index=V2_IDX)
count_v2 = es.count(index=V2_IDX)
print(f"✅ reindex 완료: {V2_IDX} ({count_v2['count']}건)")


In [ ]:
# ─── 16-C. alias 무중단 전환 ───
es.indices.update_aliases(body={
    "actions": [
        {"remove": {"index": INDEX_MAIN, "alias": ALIAS_NAME}},
        {"add":    {"index": V2_IDX,     "alias": ALIAS_NAME}}
    ]
})
print(f"✅ alias '{ALIAS_NAME}' → '{V2_IDX}'로 전환 완료 (atomic, 다운타임 없음)")
print("💡 문제 발생 시 위 액션을 reverse하면 즉시 롤백 가능합니다.")

# 정리
es.indices.delete_alias(index=V2_IDX, name=ALIAS_NAME)
es.indices.delete(index=V2_IDX)
print(f"🗑️  [{V2_IDX}] 정리 완료")


---
## §17. ES|QL Hybrid Search — FORK · FUSE · RERANK (Expert)

> ⚠️ **상태:** FORK, FUSE, RERANK는 현재 **Technical Preview** (ES 9.2+)  
> COMPLETION은 GA. 프로덕션 적용 시 Preview 상태 유의.

ES|QL의 파이프 구조는 다단계 검색 파이프라인을 **직관적으로** 표현합니다.


In [ ]:
# ─── 17-A. ES|QL 기본 Semantic Search ───
esql_17a = es.esql.query(query="""
FROM vector-search-lab METADATA _score
| WHERE content:"벡터 검색 성능을 개선하는 방법"
| KEEP title, category, _score
| SORT _score DESC
| LIMIT 5
""")
print("=== [17-A] ES|QL Semantic Search ===")
cols = [c["name"] for c in esql_17a["columns"]]
for row in esql_17a["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


In [ ]:
# ─── 17-B. ES|QL kNN 벡터 검색 ───
esql_17b = es.esql.query(query="""
FROM vector-search-lab METADATA _score
| WHERE KNN(content, "벡터 압축 기술", 5)
| KEEP title, category, _score
| SORT _score DESC
| LIMIT 5
""")
print("=== [17-B] ES|QL KNN 검색 ===")
cols = [c["name"] for c in esql_17b["columns"]]
for row in esql_17b["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


In [ ]:
# ─── 17-C. FORK + FUSE (RRF) — Hybrid Search ───
esql_17c = es.esql.query(query="""
FROM vector-search-lab METADATA _score
| FORK
    (WHERE title:"벡터 검색 최적화" | SORT _score DESC | LIMIT 10),
    (WHERE KNN(content, "벡터 검색 성능을 개선하는 방법", 10))
| FUSE
| KEEP title, category, _score
| SORT _score DESC
| LIMIT 5
""")
print("=== [17-C] FORK + FUSE (RRF) ===")
cols = [c["name"] for c in esql_17c["columns"]]
for row in esql_17c["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


In [ ]:
# ─── 17-D. FUSE normalizer=minmax (Linear 방식) ───
esql_17d = es.esql.query(query="""
FROM vector-search-lab METADATA _score
| FORK
    (WHERE title:"벡터 검색 최적화" | SORT _score DESC | LIMIT 10),
    (WHERE KNN(content, "벡터 검색 성능을 개선하는 방법", 10))
| FUSE normalizer=minmax
| KEEP title, category, _score
| SORT _score DESC
| LIMIT 5
""")
print("=== [17-D] FORK + FUSE normalizer=minmax ===")
cols = [c["name"] for c in esql_17d["columns"]]
for row in esql_17d["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


In [ ]:
# ─── 17-E. FORK + FUSE + RERANK ───
# 공식 문서 문법: RERANK "query" ON field WITH {"inference_id": "..."}
q17e = (
    "FROM vector-search-lab METADATA _score\n"
    "| FORK\n"
    '    (WHERE title:\"벡터 검색 최적화\" | SORT _score DESC | LIMIT 10),\n'
    '    (WHERE KNN(content, \"벡터 검색 성능을 개선하는 방법\", 10))\n'
    "| FUSE\n"
    f'| RERANK \"벡터 검색 성능을 개선하는 방법\" ON content WITH {{"inference_id": "{RERANK_EP}"}}\n'
    "| KEEP title, category, _score\n"
    "| LIMIT 5"
)
esql_17e = es.esql.query(query=q17e)
print("=== [17-E] FORK + FUSE + RERANK ===")
cols = [c["name"] for c in esql_17e["columns"]]
for row in esql_17e["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


In [ ]:
# ─── 17-F. RERANK — 복수 필드 조합 (CONCAT) ───
# RERANK의 ES|QL 전용 장점: 표현식을 ON 절에 사용 가능
q17f = (
    "FROM vector-search-lab METADATA _score\n"
    "| FORK\n"
    '    (WHERE title:\"벡터 검색 최적화\" | SORT _score DESC | LIMIT 10),\n'
    '    (WHERE KNN(content, \"벡터 검색 성능을 개선하는 방법\", 10))\n'
    "| FUSE\n"
    f'| RERANK \"벡터 검색 성능을 개선하는 방법\" ON CONCAT(title, \" - \", content) WITH {{"inference_id": "{RERANK_EP}"}}\n'
    "| KEEP title, category, _score\n"
    "| LIMIT 5"
)
esql_17f = es.esql.query(query=q17f)
print("=== [17-F] RERANK ON CONCAT(title, content) ===")
cols = [c["name"] for c in esql_17f["columns"]]
for row in esql_17f["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")
print("\n💡 Query DSL에서는 불가능한 ES|QL 전용 기능!")


In [ ]:
# ─── 17-G. ES|QL + Filter (pre-filter → FORK → FUSE → RERANK) ───
q17g = (
    "FROM vector-search-lab METADATA _score\n"
    "| WHERE category == \"optimization\"\n"
    "| FORK\n"
    '    (WHERE title:\"벡터 압축\" | SORT _score DESC | LIMIT 10),\n'
    '    (WHERE KNN(content, \"벡터 저장 공간을 줄이는 방법\", 10))\n'
    "| FUSE\n"
    f'| RERANK \"벡터 저장 공간을 줄이는 방법\" ON content WITH {{"inference_id": "{RERANK_EP}"}}\n'
    "| KEEP title, category, _score\n"
    "| LIMIT 5"
)
esql_17g = es.esql.query(query=q17g)
print("=== [17-G] pre-filter + FORK + FUSE + RERANK ===")
cols = [c["name"] for c in esql_17g["columns"]]
for row in esql_17g["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] {d.get('title','')} ({d.get('category','')})")


### §17-H. Query DSL vs ES|QL 비교

| 항목 | Query DSL | ES|QL |
|------|-----------|-------|
| 구조 | 중첩 JSON | 파이프 |
| 가독성 | 복잡 | 직관적 |
| 파라미터 제어 | 세밀 (rank_window_size 등) | 제한적 |
| 동적 필드 조합 | 불가 | CONCAT 등 표현식 가능 |
| 적합 | 프로덕션 API, SDK | 프로토타이핑, Discover |
| 성능 | 동일 (내부 동일 실행 경로) | 동일 |


---
## §18. 운영 확인 — Inference Endpoint 상태 점검 (Expert)


In [ ]:
# ─── 18-A. 전체 Inference Endpoint 목록 ───
all_eps = es.inference.get()
print("=== 전체 Inference Endpoint 목록 ===")
for ep in all_eps.get("endpoints", []):
    print(f"  [{ep.get('task_type')}] {ep.get('inference_id')}")


In [ ]:
# ─── 18-B. 특정 Endpoint 상태 ───
ep_detail = es.inference.get(inference_id=EMBED_EP)
print("=== Embedding Endpoint 상태 ===")
for ep in ep_detail.get("endpoints", [ep_detail]):
    print(f"  inference_id  : {ep.get('inference_id')}")
    print(f"  task_type     : {ep.get('task_type')}")
    print(f"  service       : {ep.get('service')}")
    svc = ep.get("service_settings", {})
    print(f"  model_id      : {svc.get('model_id')}")


In [ ]:
# ─── 18-C. 인덱스 설정 확인 (BBQ 적용 여부) ───
settings = es.indices.get_settings(index=INDEX_MAIN)
mapping  = es.indices.get_mapping(index=INDEX_MAIN)
print("=== 인덱스 설정 확인 ===")
print(f"인덱스: {INDEX_MAIN}")

# BBQ 적용 여부
try:
    chunks_mapping = (mapping[INDEX_MAIN]["mappings"]
                      ["properties"]["content"]
                      ["properties"]["inference"]
                      ["properties"]["chunks"]
                      ["properties"]["embeddings"])
    index_options = chunks_mapping.get("index_options", {})
    print(f"벡터 index_options: {index_options}")
except KeyError:
    print("매핑에서 embedding 정보를 찾을 수 없습니다 (임베딩 생성 대기 중일 수 있음)")


In [ ]:
# ─── 18-D. 인덱스 통계 ───
stats = es.indices.stats(index=INDEX_MAIN, metric="segments")
seg_info = stats["indices"][INDEX_MAIN]["total"]["segments"]
print("=== 인덱스 통계 ===")
print(f"  segment 수   : {seg_info.get('count', 0)}")
print(f"  메모리 사용  : {seg_info.get('memory_in_bytes', 0):,} bytes")


---
## §19. 정리 (Clean-up)

실습에서 생성한 모든 인덱스와 Inference Endpoint를 삭제합니다.  
> ⚠️ EIS 내장 endpoint(`.jina-*`)는 삭제하지 않습니다. PUT으로 생성한 커스텀 endpoint만 삭제.


In [ ]:
# ─── 19. 전체 정리 ───
INDICES_TO_DELETE = [
    "vector-search-lab",
    "vector-search-lab-custom",
    "vector-search-lab-chunk-test",
    "vector-search-lab-float32",
    "vector-search-lab-bbq",
    "vector-search-lab-v2",
    "chanbot-it-support",
]

for idx in INDICES_TO_DELETE:
    if es.indices.exists(index=idx):
        es.indices.delete(index=idx)
        print(f"🗑️  [{idx}] 삭제 완료")

print("\n✅ 인덱스 정리 완료")
print("💡 EIS 내장 endpoint(.jina-*)는 삭제할 필요 없습니다.")


---
## §20. [Applied] IT 지원 검색 시나리오 — 찬봇 맥락 (Applied Track)

Part A/B에서 학습한 Hybrid Search + Reranker를 IT 지원 도메인에 적용합니다.  
이 시나리오는 **IT Support Agent(찬봇)**의 검색 파이프라인 설계의 기초가 됩니다.


In [ ]:
# ─── 20-A. IT 지원 인덱스 생성 ───
CHANBOT_IDX = "chanbot-it-support"
recreate_index(CHANBOT_IDX, {
    "properties": {
        "question":   {"type": "text", "analyzer": "standard"},   # BM25용
        "answer":     {"type": "semantic_text", "inference_id": EMBED_EP},  # Semantic용
        "category":   {"type": "keyword"},
        "department": {"type": "keyword"}
    }
})


In [ ]:
# ─── 20-B. 샘플 IT FAQ 데이터 인덱싱 ───
FAQ_DOCS = [
    {"_id":"1","question":"VPN 접속이 안됩니다",
     "answer":"VPN 접속 실패 시 다음을 확인하세요: 1) 사내 네트워크에 연결되어 있는지 확인합니다. 2) VPN 클라이언트를 최신 버전으로 업데이트합니다. 3) 방화벽 설정에서 UDP 500, 4500 포트가 열려있는지 확인합니다. 4) IT 헬프데스크에 네트워크 장비 점검을 요청합니다.",
     "category":"network","department":"infra"},
    {"_id":"2","question":"새 노트북 세팅 방법",
     "answer":"신규 입사자 노트북 초기 세팅 절차: 1) IT 자산 관리 시스템에서 장비를 등록합니다. 2) Windows/Mac OS 기본 설정 가이드를 따릅니다. 3) 사내 보안 솔루션(EDR, DLP)을 설치합니다. 4) 개발 환경 설정은 팀별 가이드를 참조합니다. 5) VPN 및 이메일 설정을 완료합니다.",
     "category":"device","department":"it-support"},
    {"_id":"3","question":"계정 비밀번호 초기화",
     "answer":"비밀번호 초기화 방법: 1) SSO 포털에서 비밀번호 찾기를 클릭합니다. 2) 등록된 이메일로 인증 코드를 받습니다. 3) 새 비밀번호는 12자 이상, 대소문자+숫자+특수문자 포함이어야 합니다. 4) 30일마다 변경이 필요합니다. 5) 잠금된 경우 IT 보안팀에 문의합니다.",
     "category":"account","department":"security"},
    {"_id":"4","question":"Slack 봇 추가 요청",
     "answer":"Slack workspace에 새 봇을 추가하려면: 1) IT 보안 심의를 먼저 요청합니다. 2) 승인 후 Slack App Directory에서 앱을 검색합니다. 3) workspace admin이 설치를 승인합니다. 4) 봇의 권한 범위(scope)는 최소 권한 원칙을 따릅니다. 5) 외부 데이터 전송이 있는 경우 DLP 정책 검토가 필요합니다.",
     "category":"application","department":"security"},
    {"_id":"5","question":"모니터 연결 문제",
     "answer":"외장 모니터가 인식되지 않을 때: 1) 케이블 연결 상태를 확인합니다 (HDMI, USB-C, DisplayPort). 2) 디스플레이 설정에서 감지 버튼을 클릭합니다. 3) 그래픽 드라이버를 업데이트합니다. 4) 독(dock)을 사용하는 경우 독 펌웨어를 확인합니다. 5) 다른 케이블/포트로 테스트합니다.",
     "category":"device","department":"it-support"},
]

actions = [{"_index": CHANBOT_IDX, "_id": doc["_id"],
            "_source": {k:v for k,v in doc.items() if k != "_id"}}
           for doc in FAQ_DOCS]
helpers.bulk(es, actions)
print(f"✅ {len(actions)}건 인덱싱 완료. EIS 임베딩 생성 대기 중 (약 15초)...")
time.sleep(15)


In [ ]:
# ─── 20-C. Hybrid Search + Reranker 자연어 질문 ─── Query DSL
USER_QUERY = "인터넷 연결이 안돼요"

resp_chanbot = es.search(
    index=CHANBOT_IDX,
    size=3,
    source=["question", "category", "department"],
    min_score=0.3,
    retriever={
        "text_similarity_reranker": {
            "retriever": {
                "rrf": {
                    "retrievers": [
                        {"standard": {"query": {"match": {"question": USER_QUERY}}}},
                        {"standard": {"query": {"semantic": {"field": "answer", "query": USER_QUERY}}}}
                    ],
                    "rank_window_size": 50
                }
            },
            "field": "answer",
            "inference_id": RERANK_EP,
            "inference_text": USER_QUERY,
            "rank_window_size": 10
        }
    }
)
print(f"=== [Query DSL] 찬봇 질문: '{USER_QUERY}' ===")
for h in resp_chanbot["hits"]["hits"]:
    src = h['_source']
    print(f"  [{h['_score']:.4f}] Q: {src['question']} | {src['department']}/{src['category']}")
print("\n💡 '인터넷 연결' → 'VPN 접속' 문서 매칭 (의미적 유사성)")


In [ ]:
# ─── 20-C. Hybrid Search + Reranker ─── ES|QL
q20c = (
    "FROM chanbot-it-support METADATA _score\n"
    "| FORK\n"
    f'    (WHERE question:\"{USER_QUERY}\" | SORT _score DESC | LIMIT 10),\n'
    f'    (WHERE KNN(answer, \"{USER_QUERY}\", 10))\n'
    "| FUSE\n"
    f'| RERANK \"{USER_QUERY}\" ON answer WITH {{"inference_id": "{RERANK_EP}"}}\n'
    "| KEEP question, category, department, _score\n"
    "| LIMIT 3"
)
esql_chanbot = es.esql.query(query=q20c)
print(f"=== [ES|QL] 찬봇 질문: '{USER_QUERY}' ===")
cols = [c["name"] for c in esql_chanbot["columns"]]
for row in esql_chanbot["values"]:
    d = dict(zip(cols, row))
    print(f"  [{d.get('_score',0):.4f}] Q: {d.get('question','')} | {d.get('department','')}/{d.get('category','')}")


In [ ]:
# ─── 20-D. 부서 Filter + Hybrid ─── Query DSL
resp_filter_dept = es.search(
    index=CHANBOT_IDX, size=3, source=["question", "category", "department"],
    retriever={
        "rrf": {
            "retrievers": [
                {"standard": {"query": {"match": {"question": "보안 정책"}}}},
                {"standard": {"query": {"semantic": {"field": "answer", "query": "보안 관련 절차와 정책"}}}}
            ],
            "filter": {"term": {"department": "security"}},
            "rank_window_size": 50
        }
    }
)
print("=== [Query DSL] department=security filter + RRF ===")
for h in resp_filter_dept["hits"]["hits"]:
    src = h['_source']
    print(f"  [{h['_score']:.4f}] {src['question']} ({src['department']})")


In [ ]:
# ─── 20-E. BM25 vs Semantic vs Hybrid 비교 ───
compare_q = "컴퓨터가 느려요"
print(f"=== 검색 방식 비교: '{compare_q}' ===\n")

# 1) BM25만
bm25_r = es.search(index=CHANBOT_IDX, size=3, source=["question"],
                   query={"match": {"question": compare_q}})
print("1) BM25만:")
hits = bm25_r["hits"]["hits"]
print(f"   {len(hits)}건" + (" (매칭 없음)" if not hits else ""))
for h in hits: print(f"   [{h['_score']:.4f}] {h['_source']['question']}")

# 2) Semantic만
sem_r = es.search(index=CHANBOT_IDX, size=3, source=["question"],
                  query={"semantic": {"field": "answer", "query": compare_q}})
print("\n2) Semantic만:")
for h in sem_r["hits"]["hits"]: print(f"   [{h['_score']:.4f}] {h['_source']['question']}")

# 3) Hybrid (RRF)
hyb_r = es.search(index=CHANBOT_IDX, size=3, source=["question"],
                  retriever={
                      "rrf": {
                          "retrievers": [
                              {"standard": {"query": {"match": {"question": compare_q}}}},
                              {"standard": {"query": {"semantic": {"field": "answer", "query": compare_q}}}}
                          ]
                      }
                  })
print("\n3) Hybrid (RRF):")
for h in hyb_r["hits"]["hits"]: print(f"   [{h['_score']:.4f}] {h['_source']['question']}")

print("\n💡 Hybrid가 BM25 단독 대비 더 포괄적인 결과를 반환합니다.")


In [ ]:
# ─── 20-F. 찬봇 인덱스 정리 ───
es.indices.delete(index=CHANBOT_IDX)
print(f"🗑️  [{CHANBOT_IDX}] 정리 완료")
print("\n🎉 전체 실습 완료!")


---
## 부록: Quick Reference

### semantic_text vs dense_vector

| 항목 | semantic_text | dense_vector |
|------|--------------|--------------|
| Chunking | 자동 (sentence, 250w) | 직접 구현 |
| Embedding | 자동 (Inference EP) | 외부 생성 후 인덱싱 |
| HNSW 튜닝 | 제한적 | m, ef_construction 직접 제어 |
| BBQ | 자동 적용 (9.1+) | index_options으로 명시 |
| 적합 | 빠른 시작, 프로토타이핑 | 세밀한 제어, 커스텀 모델 |

### Score 결합 방식 비교

| 방식 | 수식 | 특징 | 적합 |
|------|------|------|------|
| RRF | Σ 1/(rank + k) | 순위 기반, 튜닝 불필요 | 빠른 시작 |
| Linear | w₁×norm(s₁) + w₂×norm(s₂) | score 기반, MinMax 정규화 | 프로덕션 최적화 |

### 핵심 파라미터 Quick Reference

| 파라미터 | 기본값 | 용도 |
|---------|--------|------|
| `m` (HNSW) | 16 | 노드당 연결 수 |
| `ef_construction` | 100 | 그래프 구축 탐색 범위 |
| `num_candidates` | k×1.5 | 검색 시 탐색 후보 |
| `rank_constant` (RRF) | 60 | 하위 순위 영향도 |
| `rank_window_size` | 50 | 결합 대상 문서 수 |
| `min_score` | 없음 | 품질 cutoff (RAG 필수) |

---
> **작성일:** 2026년 4월  
> **ES 버전:** 9.x  
> **모델:** `.jina-embeddings-v5-text-small` (EIS), `.jina-reranker-v2-base-multilingual` (EIS)
